# **Digit Recognizer — Start-to-Finish with usage**

This project is all about teaching a computer to read handwritten digits (0 through 9). The dataset comes from the famous MNIST digit recognition challenge, where each image is a 28×28 grayscale picture of a single handwritten number.

The goal is simple: look at an image and correctly identify which digit it is. While this might seem easy for us humans, it’s actually a classic problem in computer vision and a great way to practice image classification with deep learning.

To solve it, I built a Convolutional Neural Network (CNN) in TensorFlow/Keras. The network includes convolutional layers to detect patterns like edges and curves, batch normalization to stabilize training, dropout to prevent overfitting, and a final softmax layer to classify each digit. I also used data augmentation (small rotations, shifts, and zooms) so the model can handle variations in handwriting styles.

After training, the model was evaluated on a validation set, achieving strong accuracy. I also analyzed a confusion matrix to see which digits the model confused most often (for example, 4’s vs. 9’s). Finally, I generated predictions for the test set and submitted them to Kaggle for evaluation.

In short, this project demonstrates how deep learning can be applied to image recognition tasks. It’s a hands-on example of going from raw pixel data → model building → training → evaluation → real-world predictions.

In [ ]:
import os
import random
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # disable GPU, use CPU only

# **Reproducibility**

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# **Paths**

In [ ]:
KAGGLE_INPUT = "/kaggle/input/digit-recognizer"
TRAIN_CSV = os.path.join(KAGGLE_INPUT, "/kaggle/input/digit-recognizer/train.csv")
TEST_CSV  = os.path.join(KAGGLE_INPUT, "/kaggle/input/digit-recognizer/test.csv")
WORKING   = "/kaggle/working"

assert os.path.exists(TRAIN_CSV), "Could not find train.csv. Add the Kaggle dataset 'digit-recognizer' to your notebook."
assert os.path.exists(TEST_CSV),  "Could not find test.csv. Add the Kaggle dataset 'digit-recognizer' to your notebook."

# **Load Train and Test Data**

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

# **Split features**/**labels**

In [ ]:
y = train_df["label"].values
X = train_df.drop(columns=["label"]).values

# **Basic preprocessing**

In [ ]:
# Normalize to [0,1] and reshape to (28, 28, 1)
X = (X / 255.0).astype("float32").reshape(-1, 28, 28, 1)
X_test = (test_df.values / 255.0).astype("float32").reshape(-1, 28, 28, 1)

In [ ]:
# Train/Validation split (stratified)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.10, random_state=SEED, stratify=y
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# **Data augmentation (in-model)**

In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomRotation(0.08, fill_mode="nearest"),
        layers.RandomTranslation(0.05, 0.05, fill_mode="nearest"),
        layers.RandomZoom(0.05, 0.05),
    ],
    name="augmentation",
)

# **Model definition**

In [ ]:
def make_model():
    inputs = keras.Input(shape=(28, 28, 1))

    x = data_augmentation(inputs)

    # Block 1
    x = layers.Conv2D(32, 3, padding="same", activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(32, 3, padding="same", activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    # Block 2
    x = layers.Conv2D(64, 3, padding="same", activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(64, 3, padding="same", activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.25)(x)

    # Block 3
    x = layers.Conv2D(128, 3, padding="same", activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(128, 3, padding="same", activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Dropout(0.30)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(10, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="mnist_cnn")
    return model

model = make_model()

# **Compile**

In [ ]:
opt = keras.optimizers.Adam(learning_rate=1e-3)
model.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

# Why does it say None?
* In TensorFlow/Keras, None means “batch size not fixed”
* When you define the model, you don’t usually hardcode the batch size (e.g., 32, 64, etc.) in the input shape.
* Instead, the input layer is defined as (None, 28, 28, 1) → meaning:
    1. None = “this can be any batch size (number of images at once)”
    2. 28, 28, 1 = the actual image shape (height, width, channels).

So, if you feed 32 images, TensorFlow replaces None with 32 internally → (32, 28, 28, 1).
If you feed 256 images, it becomes (256, 28, 28, 1).

In [ ]:
inputs = keras.Input(shape=(28, 28, 1))
print(inputs.shape)   # (None, 28, 28, 1)

batch = tf.random.normal((32, 28, 28, 1))  # 32 fake images
outputs = model(batch)  
print(outputs.shape)  # (32, 10) because batch size = 32

# **Callbacks**

In [ ]:
ckpt_path = os.path.join(WORKING, "best.keras")
callbacks = [
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_accuracy", factor=0.5, patience=3, verbose=1, min_lr=1e-5
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=8, verbose=1, restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        ckpt_path, monitor="val_accuracy", save_best_only=True, verbose=1
    ),
]

# **Training**

In [ ]:
BATCH_SIZE = 256
EPOCHS = 40

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    verbose=2,
    callbacks=callbacks,
)

# **Validation performance**

In [ ]:
val_probs = model.predict(X_val, batch_size=BATCH_SIZE, verbose=0)
val_preds = np.argmax(val_probs, axis=1)
val_acc = accuracy_score(y_val, val_preds)
print(f"Validation Accuracy: {val_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_val, val_preds)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=range(10), yticklabels=range(10))
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - Validation Set")
plt.show()

In [ ]:
# Classification reportdst
print(classification_report(y_val, val_preds, digits=4))

# **Train on full data (quick fine-tune)**

In [ ]:
# Concatenate train + val and train a few more epochs starting from best weights.
# This can help you squeezing a bit more accuracy before final predictions.
if os.path.exists(ckpt_path):
    model = keras.models.load_model(ckpt_path)

history_full = model.fit(
    np.concatenate([X_train, X_val], axis=0),
    np.concatenate([y_train, y_val], axis=0),
    batch_size=BATCH_SIZE,
    epochs=5,
    verbose=2,
)

# **Predict test & create submission**

In [ ]:
test_probs = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
test_preds = np.argmax(test_probs, axis=1)

submission = pd.DataFrame({
    "ImageId": np.arange(1, len(test_preds) + 1),
    "Label": test_preds
})
sub_path = os.path.join(WORKING, "submission.csv")
submission.to_csv(sub_path, index=False)
print(f"Saved submission to: {sub_path}")

# **Save final model**


In [ ]:
final_model_path = os.path.join(WORKING, "mnist_final.keras")
model.save(final_model_path)
print(f"Saved final model to: {final_model_path}")

# **Simple sanity checks**


In [ ]:
# Show label distribution in predictions
print("Prediction distribution:", np.bincount(test_preds, minlength=10))

# **Simple Tests**


In [ ]:
import matplotlib.pyplot as plt

# Pick a few random test samplesure
indices = np.random.choice(len(X_test), 12, replace=False)
sample_images = X_test[indices]
sample_preds = test_preds[indices]

plt.figure(figsize=(10,4))
for i, (img, pred) in enumerate(zip(sample_images, sample_preds)):
    plt.subplot(2, 6, i+1)
    plt.imshow(img.squeeze(), cmap="gray")
    plt.title(f"Pred: {pred}")
    plt.axis("off")
plt.suptitle("Sample Predictions on Test Set", fontsize=16)
plt.show()


In [ ]:
# Take one validation image
idx = 5
img = X_val[idx]

plt.imshow(img.squeeze(), cmap="gray")
plt.title(f"True: {y_val[idx]}")
plt.axis("off")
plt.show()

# Model prediction
pred = np.argmax(model.predict(img.reshape(1,28,28,1)))
print("Predicted:", pred)
